# Roll No: G25AIT2067
# In-Class Activity 2 - Word probabilities with SRILM (Brown corpus)

In this activity I use the SRILM toolkit to build n-gram language models on the Brown corpus and
check their perplexity on the test set. Steps: count the n-grams, train a language model, test it,
then do the extra exercises (OOV words, different smoothing, trigram, pre-processing, interpolation).

Dataset: the Brown corpus (brown-train / brown-dev / brown-test) from the course Drive link.
To run on Colab: upload the SRILM-Lab zip from the Files panel on the left, then Runtime > Run all.

In [ ]:
# get the files ready
import os, glob, zipfile, subprocess

# if the SRILM-Lab zip was uploaded, unzip it
for z in glob.glob('/content/SRILM-Lab*.zip'):
    zipfile.ZipFile(z).extractall('/content')

# move into the folder that has ngram and ngram-count
if os.path.exists('/content/SRILM-Lab/ngram'):
    os.chdir('/content/SRILM-Lab')

os.makedirs('out', exist_ok=True)
subprocess.run('chmod +x ngram ngram-count', shell=True)
print('I am in:', os.getcwd())
print(os.listdir('browndata'))

In [ ]:
# tiny helper: run a command and print whatever it prints
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)

# run "ngram -ppl ..." and pull the perplexity number out of the output
def get_ppl(lm, test, extra=""):
    r = subprocess.run(f"./ngram {extra} -lm {lm} -ppl {test}",
                       shell=True, capture_output=True, text=True)
    out = r.stdout + r.stderr
    line = [l for l in out.splitlines() if "ppl=" in l][-1]
    return float(line.split("ppl=")[1].split("ppl1=")[0])

# pull the word count and OOV count out of the same output
def get_words_oov(lm, test, extra=""):
    r = subprocess.run(f"./ngram {extra} -lm {lm} -ppl {test}",
                       shell=True, capture_output=True, text=True)
    out = r.stdout + r.stderr
    line = [l for l in out.splitlines() if "words," in l][-1]
    words = int(line.split("sentences,")[1].split("words")[0])
    oov   = int(line.split("words,")[1].split("OOVs")[0])
    return words, oov

## Step 1 - Count the n-grams

In [ ]:
# make a bigram count file from the training data
sh("./ngram-count -text browndata/brown-train.txt -order 2 -write out/brown-train.count")
sh("wc -l out/brown-train.count")
sh("head -5 out/brown-train.count")

Checking a couple of counts by hand with grep, to be sure they match:

In [ ]:
# count of the word "the"
sh('grep -P "^the\\t" out/brown-train.count')
sh('tr " " "\\n" < browndata/brown-train.txt | grep -c "^the$"')

# number of <s> should equal the number of sentences (non-empty lines)
sh('grep -P "^<s>\\t" out/brown-train.count')
sh('grep -c . browndata/brown-train.txt')

## Step 2 - Train the language model

In [ ]:
# train an un-smoothed bigram model. -unk lets it handle unknown words with an <unk> tag
sh("./ngram-count -read out/brown-train.count -order 2 -lm out/brown-train.lm -addsmooth 0 -unk")
sh("head -8 out/brown-train.lm")

## Step 3 - Test it (perplexity and per-word probability)

In [ ]:
# perplexity on the test set
sh("./ngram -lm out/brown-train.lm -ppl browndata/brown-test.txt")

In [ ]:
# per-word probabilities for the first 2 test sentences (-debug 2).
# I only take 2 sentences here, otherwise it prints thousands of lines.
sh("head -2 browndata/brown-test.txt > out/two.txt")
sh("./ngram -lm out/brown-train.lm -ppl out/two.txt -debug 2")

## Exercise 1 - Why did OOV words get -inf? How to fix it? What do -unk and -addsmooth 0 mean?

I compare 3 bigram models to see the difference.

In [ ]:
# model 1: closed vocab (no -unk), no smoothing
sh("./ngram-count -text browndata/brown-train.txt -order 2 -lm out/m1.lm -addsmooth 0")
# model 2: -unk but still no smoothing
sh("./ngram-count -text browndata/brown-train.txt -order 2 -lm out/m2.lm -addsmooth 0 -unk")
# model 3: -unk + add-1 smoothing
sh("./ngram-count -text browndata/brown-train.txt -order 2 -lm out/m3.lm -addsmooth 1 -unk")

for name, lm in [("closed vocab, no smoothing", "out/m1.lm"),
                 ("-unk, no smoothing",         "out/m2.lm"),
                 ("-unk + add-1",               "out/m3.lm")]:
    w, oov = get_words_oov(lm, "browndata/brown-test.txt")
    print(f"{name:25s}  ppl = {get_ppl(lm, 'browndata/brown-test.txt'):8.1f}   OOVs = {oov}")

**My answer:** With -addsmooth 0 there is no smoothing, so any word or bigram that was never seen in
training gets probability 0, and log(0) is -inf. SRILM counts these as OOVs (and zeroprobs) and leaves
them out of the perplexity. -unk adds an <unk> token so unknown words have a slot, but with no smoothing
<unk> still has probability 0. So -addsmooth 0 just means "do not smooth". The fix is to use -unk
together with smoothing: add-1 gives 0 OOVs and a real perplexity over the whole test set.

## Exercise 2 - Try different smoothing methods

In [ ]:
smoothings = {
    "add-1":        "-addsmooth 1",
    "add-0.01":     "-addsmooth 0.01",
    "Good-Turing":  "",                 # this is the SRILM default
    "Witten-Bell":  "-wbdiscount",
    "Kneser-Ney":   "-kndiscount",
    "KN interp":    "-kndiscount -interpolate",
}

for name, opt in smoothings.items():
    # open vocab (-unk) so OOV words get a probability
    subprocess.run(f"./ngram-count -text browndata/brown-train.txt -order 2 {opt} -unk -lm out/s.lm", shell=True)
    # closed vocab, so every model is tested on the same words (fair comparison)
    subprocess.run(f"./ngram-count -text browndata/brown-train.txt -order 2 {opt} -lm out/sc.lm", shell=True)
    # read P(<unk>) from the open-vocab model file
    unk = [l for l in open("out/s.lm").read().splitlines() if l.split("\t")[1:2] == ["<unk>"]]
    punk = 10 ** float(unk[0].split("\t")[0]) if unk else 0
    print(f"{name:12s}  P(<unk>)={punk:.4f}   "
          f"open ppl={get_ppl('out/s.lm','browndata/brown-test.txt'):7.1f}   "
          f"closed ppl={get_ppl('out/sc.lm','browndata/brown-test.txt'):7.1f}")

**My answer:** On the same words (closed ppl) Kneser-Ney with interpolation is the best (about 394) and
add-1 is the worst (about 1881), because add-1 gives too much probability to unseen bigrams. Witten-Bell
and Kneser-Ney look best in the open column only because they give a big share of probability to <unk>
(0.05 and 0.13), which makes the roughly 6% unknown test words cheap. So perplexities can only be
compared when the vocabulary is the same.

## Exercise 3 - Trigram instead of bigram

In [ ]:
for name in ["add-1", "Good-Turing", "Witten-Bell", "KN interp"]:
    opt = smoothings[name]
    subprocess.run(f"./ngram-count -text browndata/brown-train.txt -order 2 {opt} -lm out/b2.lm", shell=True)
    subprocess.run(f"./ngram-count -text browndata/brown-train.txt -order 3 {opt} -lm out/b3.lm", shell=True)
    p2 = get_ppl("out/b2.lm", "browndata/brown-test.txt", "-order 2")
    p3 = get_ppl("out/b3.lm", "browndata/brown-test.txt", "-order 3")
    print(f"{name:12s}  bigram ppl={p2:7.1f}   trigram ppl={p3:7.1f}")

**My answer:** For the good smoothing methods the trigram is a bit better than the bigram (KN interp
393.5 to 374.3) because more context means less uncertainty. Add-1 gets worse (1881 to 2798) because
now there are even more unseen trigrams for it to waste probability on.

## Exercise 4 - Pre-processing (lower-case, remove punctuation, remove stop words)

In [ ]:
import string
stop = set("a an the is are was were be been being to of in on at for and or but if then he she it "
           "they we you i this that these those with as by from his her its their our your my".split())
punct = set(string.punctuation) | {"``", "''", "--"}

def clean(src, dst, lower=False, nopunct=False, nostop=False):
    with open(src) as f, open(dst, "w") as g:
        for line in f:
            w = line.split()
            if lower:   w = [x.lower() for x in w]
            if nopunct: w = [x for x in w if not all(c in punct for c in x)]
            if nostop:  w = [x for x in w if x.lower() not in stop]
            if w: g.write(" ".join(w) + "\n")

versions = {
    "original":                        {},
    "lower-case":                      dict(lower=True),
    "lower + no punctuation":          dict(lower=True, nopunct=True),
    "lower + no punct + no stopwords": dict(lower=True, nopunct=True, nostop=True),
}
for name, kw in versions.items():
    clean("browndata/brown-train.txt", "out/tr.txt", **kw)
    clean("browndata/brown-test.txt",  "out/te.txt", **kw)
    subprocess.run("./ngram-count -text out/tr.txt -order 2 -kndiscount -interpolate -lm out/pp.lm", shell=True)
    w, oov = get_words_oov("out/pp.lm", "out/te.txt")
    print(f"{name:32s}  OOV%={100*oov/w:5.2f}   ppl={get_ppl('out/pp.lm','out/te.txt'):8.1f}")

**My answer:** Lower-casing merges "The" and "the" so the vocabulary shrinks, OOVs drop and the ppl
goes down a little (393 to 383). Removing punctuation makes ppl go up (612) because punctuation marks
are very easy to predict. Removing stop words makes it much higher (about 3000) because the easy,
frequent words are gone and only the hard content words are left. Each version has a different set of
words, so these numbers show how predictable the text is, not which model is "better".

## Exercise 5 - Interpolate unigram + bigram, tuned on the dev set

(a) train unigram and bigram models with a small addsmooth, (b) mix them with -mix-lm and -lambda on
brown-dev, (c) find the best lambda and addsmooth, (d) test on brown-test with the best values.

In [ ]:
import pandas as pd

ks   = [0.001, 0.01, 0.1, 1]
lams = [round(0.1*i, 1) for i in range(11)]
grid = {}
for k in ks:
    subprocess.run(f"./ngram-count -text browndata/brown-train.txt -order 1 -addsmooth {k} -unk -lm out/u.lm", shell=True)
    subprocess.run(f"./ngram-count -text browndata/brown-train.txt -order 2 -addsmooth {k} -unk -lm out/b.lm", shell=True)
    for lam in lams:
        grid[(k, lam)] = get_ppl("out/b.lm", "browndata/brown-dev.txt", f"-mix-lm out/u.lm -lambda {lam}")

table = pd.Series(grid).unstack().round(1)
table.index.name = "addsmooth"; table.columns.name = "lambda (bigram weight)"
table

In [ ]:
# best on dev, then test with it
best_k, best_lam = min(grid, key=grid.get)
print(f"best: addsmooth={best_k}, lambda={best_lam}, dev ppl={grid[(best_k, best_lam)]:.1f}")

subprocess.run(f"./ngram-count -text browndata/brown-train.txt -order 1 -addsmooth {best_k} -unk -lm out/u.lm", shell=True)
subprocess.run(f"./ngram-count -text browndata/brown-train.txt -order 2 -addsmooth {best_k} -unk -lm out/b.lm", shell=True)

print("unigram only   test ppl =", round(get_ppl("out/u.lm", "browndata/brown-test.txt"), 1))
print("bigram only    test ppl =", round(get_ppl("out/b.lm", "browndata/brown-test.txt"), 1))
print("interpolated   test ppl =", round(get_ppl("out/b.lm", "browndata/brown-test.txt",
                                    f"-mix-lm out/u.lm -lambda {best_lam}"), 1))

**My answer:** The best dev perplexity is about 581 at addsmooth=0.001 and lambda=0.7 (so 70% weight on
the bigram). On the unseen test set the interpolated model gets about 681, better than the bigram alone
(about 963) and the unigram alone (about 1363). Interpolation helps because when a bigram is rare or
unseen, the model can fall back on the unigram. (Note: the flag is -lambda, the handout writes -lamda.)

## Summary

In [ ]:
print("Lower perplexity means a better model.")
print("Best model here: Kneser-Ney with interpolation  ->  bigram about 394, trigram about 374.")